Các phân tích trong đây phục vụ để có được insight và sử dụng chủ yếu cho Dashboard số 1. Không phải tất cả các thay đổi trong Notebooks này sẽ được sử dụng để train mô hình do vấn đề Data Leakage, chỉ một số các thay đổi được đánh giá là an toàn mới được sử dụng. Các thay đổi đó được xem xét thận trọng và fit chỉ trên tập train, perform trên tập val, test; sẽ được tiến hành sau quá trình train_test_split

Thay đổi bao gồm: 
1. Chuẩn hóa fullVisitorId và thời gian
2. Xác định revenue columns hiện có
3. Ép revenue về numeric
4. Kiểm tra missing, âm, outlier, zero/non-zero
5. Tính target chuẩn theo đề:
   target_log_revenue = log(1 + tổng transactionRevenue theo fullVisitorId)
6. Group theo fullVisitorId
7. In thống kê mô tả đầy đủ
8. So sánh target cũ nếu dataframe đã có target_log_revenue

| Cột                              | Đơn vị                       | Ý nghĩa                                                                                       |
| -------------------------------- | ---------------------------- | --------------------------------------------------------------------------------------------- |
| `transactionRevenue`             | **micro-dollar**             | Doanh thu gốc từ GA, phải chia `1_000_000` để ra dollar                                       |
| `totals_transactionRevenue`      | **micro-dollar**             | Giống `transactionRevenue`; bạn đã kiểm tra thấy 2 cột này khớp 100%                          |
| `totals_totalTransactionRevenue` | **micro-dollar**             | Tổng revenue/session theo GA, cũng ở dạng micro-dollar nhưng có thể khác `transactionRevenue` |
| `transactionRevenue_dollar`      | **dollar**                   | Đã được tạo bằng `transactionRevenue / 1_000_000`                                             |
| `revenue_raw`                    | **micro-dollar**             | Bản copy từ `transactionRevenue`                                                              |
| `revenue_dollar`                 | **dollar**                   | Bản copy từ `transactionRevenue_dollar`                                                       |
| `total_revenue_raw`              | **micro-dollar**             | Tổng `transactionRevenue` theo `fullVisitorId`                                                |
| `total_revenue_dollar`           | **dollar**                   | Tổng `transactionRevenue_dollar` theo `fullVisitorId`                                         |
| `target_log_revenue`             | **log của micro-dollar + 1** | `log1p(total_revenue_raw)` nếu tính đúng theo user                                            |
| `target_log_revenue_dollar`      | **log của dollar + 1**       | `log1p(total_revenue_dollar)`                                                                 |


# Hướng dẫn lấy dữ liệu cho Dashboard 1

1. Cohort Analysis

| Mục đích       | Cột nên dùng                          | Vì sao                                                                |
| -------------- | ------------------------------------- | --------------------------------------------------------------------- |
| ID khách hàng  | `fullVisitorId`                       | Gom session theo user                                                 |
| ID session     | `visitId`                             | Đếm số phiên                                                          |
| Ngày truy cập  | `date` hoặc `visitStartTime_datetime` | Xác định cohort month, return month                                   |
| Thứ tự phiên   | `visitNumber`                         | Biết user mới/quay lại                                                |
| Revenue raw    | `transactionRevenue`                  | Tính doanh thu theo cohort                                            |
| Revenue dollar | `transactionRevenue_dollar`           | Vẽ dashboard dễ hiểu                                                  |
| Target log     | `target_log_revenue`                  | Chỉ dùng nếu đã tính đúng theo user/window; không dùng cho cohort thô |


2. Cohort acquisition

| Mục đích      | Cột                          |
| ------------- | ---------------------------- |
| Kênh tổng     | `channelGrouping`            |
| Source        | `trafficSource_source`       |
| Medium        | `trafficSource_medium`       |
| Campaign      | `trafficSource_campaign`     |
| Keyword       | `trafficSource_keyword`      |
| Referral path | `trafficSource_referralPath` |
| True direct   | `trafficSource_isTrueDirect` |


3. Engagement

| Metric          | Cột                            |
| --------------- | ------------------------------ |
| Số phiên        | `visitId` hoặc `totals_visits` |
| Hits            | `totals_hits`                  |
| Pageviews       | `totals_pageviews`             |
| Bounce          | `totals_bounces`               |
| New visits      | `totals_newVisits`             |
| Time on site    | `totals_timeOnSite`            |
| Session quality | `totals_sessionQualityDim`     |


4. Geo/device dashboard

| Dashboard           | Cột                      |
| ------------------- | ------------------------ |
| Country performance | `geoNetwork_country`     |
| City performance    | `geoNetwork_city`        |
| Region              | `geoNetwork_region`      |
| Continent           | `geoNetwork_continent`   |
| Device category     | `device_deviceCategory`  |
| Browser             | `device_browser`         |
| Operating system    | `device_operatingSystem` |
| Mobile vs desktop   | `device_isMobile`        |


In [10]:
import pandas as pd
import numpy as np

# Load lại data từ file pickle
df_work = pd.read_pickle(r"D:\data_driven_marketing\df_full.pkl")

# Kiểm tra nhanh
print(df_work.shape)
df_work.head()
# 0. Copy data
# ============================================================

print("=" * 80)
print("0. BASIC DATA INFO")
print("=" * 80)

print("Shape:", df_work.shape)
print("Number of columns:", df_work.shape[1])
print("Columns:")
print(df_work.columns.tolist())

(1708337, 59)
0. BASIC DATA INFO
Shape: (1708337, 59)
Number of columns: 59
Columns:
['channelGrouping', 'date', 'fullVisitorId', 'visitId', 'visitNumber', 'visitStartTime', 'device_browser', 'device_operatingSystem', 'device_isMobile', 'device_deviceCategory', 'geoNetwork_continent', 'geoNetwork_subContinent', 'geoNetwork_country', 'geoNetwork_region', 'geoNetwork_metro', 'geoNetwork_city', 'geoNetwork_networkDomain', 'totals_visits', 'totals_hits', 'totals_pageviews', 'totals_bounces', 'totals_newVisits', 'totals_sessionQualityDim', 'totals_timeOnSite', 'totals_transactions', 'totals_transactionRevenue', 'totals_totalTransactionRevenue', 'trafficSource_campaign', 'trafficSource_source', 'trafficSource_medium', 'trafficSource_keyword', 'trafficSource_referralPath', 'trafficSource_isTrueDirect', 'trafficSource_adContent', 'trafficSource_adwordsClickInfo.page', 'trafficSource_adwordsClickInfo.slot', 'trafficSource_adwordsClickInfo.gclId', 'trafficSource_adwordsClickInfo.adNetworkType', 

In [11]:
df_work.describe()

,date,visitId,visitNumber,visitStartTime,totals_visits,totals_hits,totals_pageviews,totals_bounces,totals_newVisits,totals_sessionQualityDim,...,Date_Is_month_start,Date_Is_quarter_end,Date_Is_quarter_start,Date_Is_year_end,Date_Is_year_start,visitStartTime_datetime,Date_Hour,transactionRevenue,transactionRevenue_dollar,target_log_revenue
count,1708337,1.708337e+06,1.708337e+06,1.708337e+06,1708337.0,1.708337e+06,1.708337e+06,1.708337e+06,1.708337e+06,1.708337e+06,...,1.708337e+06,1.708337e+06,1.708337e+06,1.708337e+06,1.708337e+06,1708337,1.708337e+06,1.708337e+06,1.708337e+06,1.708337e+06
mean,2017-06-24 05:47:18.118005504,1.498352e+09,2.335170e+00,1.498352e+09,1.0,4.429598e+00,3.695685e+00,5.101909e-01,7.653232e-01,1.915779e+00,...,3.141418e-02,8.343202e-03,8.219690e-03,1.583997e-03,1.857947e-03,2017-06-25 00:57:35.545791488,1.245210e+01,1.355906e+06,1.355906e+00,1.925875e-01
min,2016-08-01 00:00:00,1.470035e+09,1.000000e+00,1.470035e+09,1.0,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2016-08-01 07:00:12,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2016-12-25 00:00:00,1.482738e+09,1.000000e+00,1.482738e+09,1.0,1.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2016-12-26 07:38:16,7.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2017-07-11 00:00:00,1.499832e+09,1.000000e+00,1.499832e+09,1.0,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2017-07-12 03:56:53,1.400000e+01,0.000000e+00,0.000000e+00,0.000000e+00
75%,2017-12-05 00:00:00,1.512513e+09,1.000000e+00,1.512513e+09,1.0,4.000000e+00,4.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2017-12-05 22:33:42,1.800000e+01,0.000000e+00,0.000000e+00,0.000000e+00
max,2018-04-30 00:00:00,1.525158e+09,4.570000e+02,1.525158e+09,1.0,5.000000e+02,5.000000e+02,1.000000e+00,1.000000e+00,1.000000e+02,...,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,2018-05-01 06:56:58,2.300000e+01,2.312950e+10,2.312950e+04,2.386437e+01
std,NaN,1.624937e+07,9.354034e+00,1.624937e+07,0.0,8.991748e+00,6.472932e+00,4.998963e-01,4.237968e-01,8.334279e+00,...,1.744344e-01,9.095932e-02,9.028915e-02,3.976793e-02,4.306386e-02,NaN,6.891832e+00,4.522809e+07,4.522809e+01,1.844057e+00


In [12]:
# ============================================================
# 1. Required columns check
# ============================================================
print("\n" + "=" * 80)
print("1. REQUIRED COLUMNS CHECK")
print("=" * 80)

required_cols = ["fullVisitorId"]
missing_required = [c for c in required_cols if c not in df_work.columns]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

df_work["fullVisitorId"] = df_work["fullVisitorId"].astype(str)

print("fullVisitorId dtype:", df_work["fullVisitorId"].dtype)
print("Unique fullVisitorId:", df_work["fullVisitorId"].nunique())
print("Total rows:", len(df_work))


1. REQUIRED COLUMNS CHECK
fullVisitorId dtype: object
Unique fullVisitorId: 1323730
Total rows: 1708337


In [13]:
# ============================================================
# 2. Date / time normalization
# ============================================================
print("\n" + "=" * 80)
print("2. DATE / TIME CHECK")
print("=" * 80)

if "visitStartTime_datetime" in df_work.columns:
    df_work["event_date"] = pd.to_datetime(df_work["visitStartTime_datetime"], errors="coerce")
elif "date" in df_work.columns:
    df_work["event_date"] = pd.to_datetime(df_work["date"], errors="coerce")
elif "visitStartTime" in df_work.columns:
    df_work["event_date"] = pd.to_datetime(df_work["visitStartTime"], unit="s", errors="coerce")
else:
    df_work["event_date"] = pd.NaT
    print("WARNING: No date column found.")

print("event_date missing:", df_work["event_date"].isna().sum())
print("Min date:", df_work["event_date"].min())
print("Max date:", df_work["event_date"].max())


2. DATE / TIME CHECK
event_date missing: 0
Min date: 2016-08-01 07:00:12
Max date: 2018-05-01 06:56:58


In [14]:
# ============================================================
# 3. Revenue columns detection
# ============================================================
print("\n" + "=" * 80)
print("3. REVENUE COLUMNS DETECTION")
print("=" * 80)

possible_revenue_cols = [
    "transactionRevenue",
    "transactionRevenue_dollar",
    "totals_transactionRevenue",
    "totals_totalTransactionRevenue",
    "target_log_revenue"
]

existing_revenue_cols = [c for c in possible_revenue_cols if c in df_work.columns]

print("Existing revenue-related columns:")
print(existing_revenue_cols)

if len(existing_revenue_cols) == 0:
    raise ValueError("No revenue-related columns found.")

for col in existing_revenue_cols:
    df_work[col] = pd.to_numeric(df_work[col], errors="coerce")

print("\nRevenue dtypes after numeric conversion:")
print(df_work[existing_revenue_cols].dtypes)



3. REVENUE COLUMNS DETECTION
Existing revenue-related columns:
['transactionRevenue', 'transactionRevenue_dollar', 'totals_transactionRevenue', 'totals_totalTransactionRevenue', 'target_log_revenue']

Revenue dtypes after numeric conversion:
transactionRevenue                float64
transactionRevenue_dollar         float64
totals_transactionRevenue         float64
totals_totalTransactionRevenue    float64
target_log_revenue                float64
dtype: object


In [15]:
# ============================================================
# 4. Choose canonical revenue source
# ============================================================
print("\n" + "=" * 80)
print("4. CHOOSE CANONICAL REVENUE SOURCE")
print("=" * 80)

if "transactionRevenue" in df_work.columns:
    revenue_col = "transactionRevenue"
elif "totals_transactionRevenue" in df_work.columns:
    revenue_col = "totals_transactionRevenue"
elif "totals_totalTransactionRevenue" in df_work.columns:
    revenue_col = "totals_totalTransactionRevenue"
else:
    raise ValueError(
        "No valid raw revenue column found. Only target_log_revenue exists, cannot reconstruct raw revenue safely."
    )

print("Canonical revenue column used:", revenue_col)

df_work[revenue_col] = df_work[revenue_col].fillna(0)
df_work["revenue_raw"] = df_work[revenue_col]

if "transactionRevenue_dollar" in df_work.columns:
    df_work["revenue_dollar"] = df_work["transactionRevenue_dollar"].fillna(0)
else:
    df_work["revenue_dollar"] = np.nan

print("Created columns: revenue_raw, revenue_dollar")



4. CHOOSE CANONICAL REVENUE SOURCE
Canonical revenue column used: transactionRevenue
Created columns: revenue_raw, revenue_dollar


In [16]:
# ============================================================
# 5. Revenue quality checks
# ============================================================
print("\n" + "=" * 80)
print("5. REVENUE QUALITY CHECKS")
print("=" * 80)

def revenue_quality_report(data, col):
    s = data[col]
    return {
        "column": col,
        "dtype": str(s.dtype),
        "n_rows": len(s),
        "n_missing": int(s.isna().sum()),
        "missing_rate": float(s.isna().mean()),
        "n_zero": int((s.fillna(0) == 0).sum()),
        "zero_rate": float((s.fillna(0) == 0).mean()),
        "n_positive": int((s.fillna(0) > 0).sum()),
        "positive_rate": float((s.fillna(0) > 0).mean()),
        "n_negative": int((s.fillna(0) < 0).sum()),
        "negative_rate": float((s.fillna(0) < 0).mean()),
        "min": float(s.min()) if s.notna().any() else np.nan,
        "max": float(s.max()) if s.notna().any() else np.nan,
        "mean": float(s.mean()) if s.notna().any() else np.nan,
        "median": float(s.median()) if s.notna().any() else np.nan,
        "std": float(s.std()) if s.notna().any() else np.nan,
        "n_unique": int(s.nunique(dropna=True))
    }

quality_df = pd.DataFrame([
    revenue_quality_report(df_work, col)
    for col in existing_revenue_cols
])

display(quality_df)

print("\nDescriptive statistics for revenue_raw:")
display(df_work["revenue_raw"].describe(percentiles=[
    0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99
]))

print("\nTop 20 largest revenue_raw rows:")
show_cols = ["fullVisitorId", "event_date", "revenue_raw"]
if "transactionRevenue_dollar" in df_work.columns:
    show_cols.append("transactionRevenue_dollar")

display(
    df_work.sort_values("revenue_raw", ascending=False)[show_cols].head(20)
)



5. REVENUE QUALITY CHECKS


,column,dtype,n_rows,n_missing,missing_rate,n_zero,zero_rate,n_positive,positive_rate,n_negative,negative_rate,min,max,mean,median,std,n_unique
0,transactionRevenue,float64,1708337,0,0.0,1689823,0.989163,18514,0.010837,0,0.0,0.0,2.312950e+10,1.355906e+06,0.0,4.522809e+07,7252
1,transactionRevenue_dollar,float64,1708337,0,0.0,1689823,0.989163,18514,0.010837,0,0.0,0.0,2.312950e+04,1.355906e+00,0.0,4.522809e+01,7252
2,totals_transactionRevenue,float64,1708337,0,0.0,1689823,0.989163,18514,0.010837,0,0.0,0.0,2.312950e+10,1.355906e+06,0.0,4.522809e+07,7252
3,totals_totalTransactionRevenue,float64,1708337,0,0.0,1689823,0.989163,18514,0.010837,0,0.0,0.0,4.708206e+10,1.547767e+06,0.0,6.881097e+07,8507
4,target_log_revenue,float64,1708337,0,0.0,1689823,0.989163,18514,0.010837,0,0.0,0.0,2.386437e+01,1.925875e-01,0.0,1.844057e+00,7252



Descriptive statistics for revenue_raw:


count    1.708337e+06
mean     1.355906e+06
std      4.522809e+07
min      0.000000e+00
1%       0.000000e+00
5%       0.000000e+00
10%      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
90%      0.000000e+00
95%      0.000000e+00
99%      1.231000e+07
max      2.312950e+10
Name: revenue_raw, dtype: float64


Top 20 largest revenue_raw rows:


,fullVisitorId,event_date,revenue_raw,transactionRevenue_dollar
1361871,1957458976293878100,2017-04-05 20:19:40,2.312950e+10,23129.50
1472897,1957458976293878100,2017-02-14 18:30:28,1.785550e+10,17855.50
1376484,5632276788326171571,2016-09-16 14:20:43,1.602375e+10,16023.75
220100,1957458976293878100,2017-09-13 13:56:28,1.322940e+10,13229.40
1518403,1957458976293878100,2017-08-23 15:12:37,1.229300e+10,12293.00
649340,9417857471295131045,2017-07-18 19:00:09,1.058914e+10,10589.14
516657,1957458976293878100,2017-08-02 17:20:28,9.925110e+09,9925.11
893239,1957458976293878100,2017-03-24 18:36:00,8.677830e+09,8677.83
549475,4471415710206918415,2017-04-27 13:31:56,8.248800e+09,8248.80
979065,2050125810100188362,2017-09-01 19:23:06,7.427430e+09,7427.43


In [17]:
# ============================================================
# 6. Revenue column consistency
# ============================================================
print("\n" + "=" * 80)
print("6. REVENUE COLUMN CONSISTENCY CHECK")
print("=" * 80)

if {"transactionRevenue", "totals_transactionRevenue"}.issubset(df_work.columns):
    tmp = df_work[["transactionRevenue", "totals_transactionRevenue"]].fillna(0).copy()
    diff = (tmp["transactionRevenue"] - tmp["totals_transactionRevenue"]).abs()

    print("transactionRevenue vs totals_transactionRevenue")
    print("Rows different:", int((diff > 0).sum()))
    print("Different rate:", float((diff > 0).mean()))
    print("Max absolute diff:", diff.max())

    if (diff > 0).sum() > 0:
        print("\nSample different rows:")
        display(
            df_work.loc[diff > 0, [
                "fullVisitorId",
                "event_date",
                "transactionRevenue",
                "totals_transactionRevenue"
            ]].head(20)
        )

if "transactionRevenue_dollar" in df_work.columns:
    dollar_positive = df_work["transactionRevenue_dollar"].fillna(0) > 0
    raw_positive = df_work["revenue_raw"].fillna(0) > 0

    print("\nRaw revenue positive rows:", int(raw_positive.sum()))
    print("Dollar revenue positive rows:", int(dollar_positive.sum()))
    print("Rows where raw positive but dollar zero:", int((raw_positive & ~dollar_positive).sum()))
    print("Rows where dollar positive but raw zero:", int((dollar_positive & ~raw_positive).sum()))

    if raw_positive.sum() > 0 and dollar_positive.sum() > 0:
        ratio = (
            df_work.loc[raw_positive & dollar_positive, "revenue_raw"] /
            df_work.loc[raw_positive & dollar_positive, "transactionRevenue_dollar"]
        )
        print("\nRaw / dollar ratio description:")
        display(ratio.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))




6. REVENUE COLUMN CONSISTENCY CHECK
transactionRevenue vs totals_transactionRevenue
Rows different: 0
Different rate: 0.0
Max absolute diff: 0.0

Raw revenue positive rows: 18514
Dollar revenue positive rows: 18514
Rows where raw positive but dollar zero: 0
Rows where dollar positive but raw zero: 0

Raw / dollar ratio description:


count    1.851400e+04
mean     1.000000e+06
std      6.368346e-11
min      1.000000e+06
1%       1.000000e+06
5%       1.000000e+06
50%      1.000000e+06
95%      1.000000e+06
99%      1.000000e+06
max      1.000000e+06
dtype: float64

In [18]:
# =============================
# USER-LEVEL (CORE VERSION)
# =============================

df = df_work.copy()

# --- Basic cleaning ---
df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")
df["revenue_raw"] = pd.to_numeric(df["revenue_raw"], errors="coerce").fillna(0)
df["has_revenue"] = (df["revenue_raw"] > 0).astype(int)

# --- Aggregate user level ---
user = (
    df.sort_values(["fullVisitorId", "event_date"])
      .groupby("fullVisitorId", as_index=False)
      .agg(
          first_date=("event_date", "min"),
          last_date=("event_date", "max"),
          n_sessions=("visitId", "nunique"),
          total_revenue=("revenue_raw", "sum"),
          n_revenue_sessions=("has_revenue", "sum")
      )
)

# =============================
# COHORT + RETENTION
# =============================

user["cohort_month"] = user["first_date"].dt.to_period("M").astype(str)
user["last_active_month"] = user["last_date"].dt.to_period("M").astype(str)

user["lifetime_days"] = (user["last_date"] - user["first_date"]).dt.days

user["retained_7d"]  = (user["lifetime_days"] >= 7).astype(int)
user["retained_30d"] = (user["lifetime_days"] >= 30).astype(int)

user["is_returning"] = (user["n_sessions"] > 1).astype(int)

# =============================
# RFM (IMPORTANT CHO MODEL)
# =============================

snapshot_date = df["event_date"].max()

user["recency"] = (snapshot_date - user["last_date"]).dt.days
user["frequency"] = user["n_sessions"]
user["monetary"] = user["total_revenue"]

user["target_log_revenue"] = np.log1p(user["total_revenue"])
user["is_buyer"] = (user["total_revenue"] > 0).astype(int)

# =============================
# OPTIONAL nhưng hữu ích
# =============================

user["conversion_rate"] = (
    user["n_revenue_sessions"] / user["n_sessions"].replace(0, np.nan)
)

user["avg_order_value"] = np.where(
    user["n_revenue_sessions"] > 0,
    user["total_revenue"] / user["n_revenue_sessions"],
    0
)

print(user.shape)
user.head()

(1323730, 19)


,fullVisitorId,first_date,last_date,n_sessions,total_revenue,n_revenue_sessions,cohort_month,last_active_month,lifetime_days,retained_7d,retained_30d,is_returning,recency,frequency,monetary,target_log_revenue,is_buyer,conversion_rate,avg_order_value
0,0000000259678714014,2017-11-28 23:33:21,2017-11-29 00:19:40,2,0.0,0,2017-11,2017-11,0,0,0,1,153,2,0.0,0.0,0,0.0,0.0
1,0000010278554503158,2016-10-21 05:57:46,2016-10-21 05:57:46,1,0.0,0,2016-10,2016-10,0,0,0,0,557,1,0.0,0.0,0,0.0,0.0
2,0000020424342248747,2016-12-01 07:55:01,2016-12-01 07:55:01,1,0.0,0,2016-12,2016-12,0,0,0,0,515,1,0.0,0.0,0,0.0,0.0
3,0000027376579751715,2017-02-12 02:24:53,2017-02-12 02:24:53,1,0.0,0,2017-02,2017-02,0,0,0,0,443,1,0.0,0.0,0,0.0,0.0
4,0000039460501403861,2017-03-27 15:45:16,2017-03-27 15:45:16,1,0.0,0,2017-03,2017-03,0,0,0,0,399,1,0.0,0.0,0,0.0,0.0


In [19]:
user.to_pickle(r"D:\data_driven_marketing\user_level_dataset.pkl")

In [20]:
user.describe()

,first_date,last_date,n_sessions,total_revenue,n_revenue_sessions,lifetime_days,retained_7d,retained_30d,is_returning,recency,frequency,monetary,target_log_revenue,is_buyer,conversion_rate,avg_order_value
count,1323730,1323730,1.323730e+06,1.323730e+06,1.323730e+06,1.323730e+06,1.323730e+06,1.323730e+06,1.323730e+06,1.323730e+06,1.323730e+06,1.323730e+06,1.323730e+06,1.323730e+06,1.323730e+06,1.323730e+06
mean,2017-06-19 16:52:30.191542016,2017-06-22 23:26:27.350277888,1.289246e+00,1.749862e+06,1.398624e-02,3.218905e+00,5.707508e-02,2.606347e-02,1.396199e-01,3.118025e+02,1.289246e+00,1.749862e+06,2.168714e-01,1.219357e-02,6.633470e-03,1.280748e+06
min,2016-08-01 07:00:12,2016-08-01 07:00:12,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2016-12-16 05:57:11.249999872,2016-12-19 19:41:51,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.460000e+02,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2017-07-04 02:02:46,2017-07-07 21:09:53.500000,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.970000e+02,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,2017-12-02 05:13:12.249999872,2017-12-06 06:33:45.750000128,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.970000e+02,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,2018-05-01 06:56:58,2018-05-01 06:56:58,4.000000e+02,1.198516e+11,3.300000e+01,6.340000e+02,1.000000e+00,1.000000e+00,1.000000e+00,6.370000e+02,4.000000e+02,1.198516e+11,2.550952e+01,1.000000e+00,2.000000e+00,1.602375e+10
std,NaN,NaN,1.563996e+00,1.153996e+08,1.461098e-01,2.214533e+01,2.319861e-01,1.593242e-01,3.465923e-01,1.892357e+02,1.563996e+00,1.153996e+08,1.956564e+00,1.097493e-01,7.019589e-02,3.177956e+07


In [21]:
import json
from pathlib import Path

# =========================
# 1. Đường dẫn notebook hiện tại
# =========================
# Sửa tên file .ipynb của bạn tại đây
notebook_path = Path("Target_col_analysis.ipynb")

# File output txt
output_txt_path = Path("all_notebook_outputs.txt")

# =========================
# 2. Đọc file ipynb
# =========================
with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

all_outputs = []

# =========================
# 3. Duyệt từng cell và lấy output
# =========================
for i, cell in enumerate(nb.get("cells", []), start=1):
    cell_type = cell.get("cell_type", "")

    if cell_type != "code":
        continue

    outputs = cell.get("outputs", [])

    if not outputs:
        continue

    all_outputs.append("=" * 100)
    all_outputs.append(f"CELL {i}")
    all_outputs.append("=" * 100)

    source_code = "".join(cell.get("source", []))
    all_outputs.append("\n[CODE]\n")
    all_outputs.append(source_code)

    all_outputs.append("\n[OUTPUT]\n")

    for output in outputs:
        output_type = output.get("output_type", "")

        # print(), stdout, stderr
        if output_type == "stream":
            text = output.get("text", "")
            if isinstance(text, list):
                text = "".join(text)
            all_outputs.append(text)

        # errors / traceback
        elif output_type == "error":
            ename = output.get("ename", "")
            evalue = output.get("evalue", "")
            traceback = output.get("traceback", [])

            all_outputs.append(f"{ename}: {evalue}\n")
            all_outputs.append("\n".join(traceback))

        # display(), dataframe text/plain, plots metadata, etc.
        elif output_type in ["execute_result", "display_data"]:
            data = output.get("data", {})

            if "text/plain" in data:
                text = data["text/plain"]
                if isinstance(text, list):
                    text = "".join(text)
                all_outputs.append(text)

            elif "text/html" in data:
                html = data["text/html"]
                if isinstance(html, list):
                    html = "".join(html)
                all_outputs.append("[HTML OUTPUT]\n")
                all_outputs.append(html)

            elif "image/png" in data:
                all_outputs.append("[IMAGE OUTPUT: image/png omitted]\n")

            else:
                all_outputs.append(f"[UNSUPPORTED DISPLAY DATA KEYS: {list(data.keys())}]\n")

    all_outputs.append("\n\n")

# =========================
# 4. Ghi ra file txt
# =========================
with open(output_txt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(all_outputs))

print(f"Done. Saved notebook outputs to: {output_txt_path.resolve()}")

Done. Saved notebook outputs to: D:\data_driven_marketing\Notebooks\all_notebook_outputs.txt
